# Point defect model

A B2 compound is stable over a range of compositions, because atoms can swap between its two sublattices or leave vacancies behind. In landau you describe this with the energy of each defect, which you can get directly from DFT supercell calculations.

Here the compound competes with solid solutions on both sides and a liquid. The model is the one from the [landau point defect example](https://github.com/eisenforschung/landau/blob/main/notebooks/PointDefects.ipynb).

Saves `Point_Defects_Exact.svg`/`.png`.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from landau import IdealSolution, LinePhase, plot_phase_diagram
from landau.calculate import calc_phase_diagram
from landau.phases import kB
from landau.phases.pointdefects import ConstantPointDefect, PointDefectedPhase, PointDefectSublattice

In [ ]:
OUT = "Point_Defects_Exact.svg"

FIGSIZE = (6.4, 4.8)
FONTSIZE = 13
DPI = 200

## B2 compound

A stoichiometric AB host with one A and one B sublattice, each holding half of the sites. Every sublattice gets an antisite and a vacancy. $[n_i]$ is how many B atoms the defect adds.

| Defect | Sublattice | $E_i$ (eV) | $[n_i]$ |
|--------|------------|-----------|--------|
| B$_\alpha$ antisite | $\alpha$ (A sites) | 0.28 | +1 |
| V$_\alpha$ vacancy  | $\alpha$ (A sites) | 0.45 | +½ |
| A$_\beta$ antisite  | $\beta$ (B sites)  | 0.30 | −1 |
| V$_\beta$ vacancy   | $\beta$ (B sites)  | 0.50 | −½ |

In [ ]:
host = LinePhase("AB", fixed_concentration=0.5, line_energy=-0.40, line_entropy=1.0 * kB)

B_alpha = ConstantPointDefect("B_alpha", excess_energy=0.28, excess_entropy=0, excess_solutes=+1)
V_alpha = ConstantPointDefect("V_alpha", excess_energy=0.45, excess_entropy=0, excess_solutes=+0.5)
A_beta = ConstantPointDefect("A_beta", excess_energy=0.30, excess_entropy=0, excess_solutes=-1)
V_beta = ConstantPointDefect("V_beta", excess_energy=0.50, excess_entropy=0, excess_solutes=-0.5)

eta = 0.5  # sites per atom on each sublattice
ab_exact = PointDefectedPhase(
    name="AB",
    line_phase=host,
    sublattices=[
        PointDefectSublattice(name="alpha", sublattice=0, sublattice_fraction=eta, defects=[B_alpha, V_alpha]),
        PointDefectSublattice(name="beta", sublattice=1, sublattice_fraction=eta, defects=[A_beta, V_beta]),
    ],
)

## Competing phases

Solid solutions at both ends, and a liquid at high temperature.

In [ ]:
solid = IdealSolution("solid",
    LinePhase("A", fixed_concentration=0, line_energy=-0.33, line_entropy=1.0 * kB),
    LinePhase("B", fixed_concentration=1, line_energy=-0.30, line_entropy=1.0 * kB))
liquid = IdealSolution("liquid",
    LinePhase("A(l)", fixed_concentration=0, line_energy=-0.10, line_entropy=3.3 * kB),
    LinePhase("B(l)", fixed_concentration=1, line_energy=-0.13, line_entropy=3.3 * kB))

## Phase diagram

In [ ]:
df = calc_phase_diagram([solid, liquid, ab_exact], Ts=np.linspace(200, 1400, 25), mu=np.linspace(-0.8, 0.8, 200))

In [ ]:
plt.style.use("default")
plt.rcParams.update({"font.size": FONTSIZE, "svg.fonttype": "none"})
fig, ax = plt.subplots(figsize=FIGSIZE, layout="constrained")

plot_phase_diagram(df, triplepoints=True, transition_temperatures=True, ax=ax)
ax.set_xlim(0, 1)
ax.set_ylim(200, 1400)
ax.set_xlabel("$c_\\mathrm{B}$")

fig.savefig(OUT)
fig.savefig(OUT.replace(".svg", ".png"), dpi=DPI, transparent=True)

## Defect populations

landau also gives the fraction of sites holding each defect. Scanning the chemical potential at 1000 K shows how antisites take over away from stoichiometry.

In [ ]:
dmu = np.linspace(-1.0, 1.0, 400)
fractions = ab_exact.site_fractions(1000, dmu)
c = ab_exact.concentration(1000, dmu)

fig, ax = plt.subplots(figsize=FIGSIZE, layout="constrained")
for sublattice, defects in fractions.items():
    for name, fraction in defects.items():
        ax.plot(c, fraction, label=name)
ax.set_xlim(0, 1)
ax.set_yscale("log")
ax.set_ylim(1e-6, 2)
ax.set_xlabel("$c_\\mathrm{B}$")
ax.set_ylabel("defect site fraction")
ax.legend();